# 06｜Quant XGBoost、baseline 與歷史 OOS（Stage 05.1 擴充特徵版）

**執行環境：Google Colab（Python 3）**。會讀取 `janusChatGPT` 研究資料及使用 GitHub `main` 的**固定 SHA**，結果只寫入研究 run／報告目錄。不會修改來源 canonical、Risk ACTIVE、GCP、Registry、PG 或 Core/Mart。

請由上往下執行每一個 cell；首次使用先完成 Google Drive OAuth。

**訓練契約**：使用 Stage 05.1 的擴充 X（quant 15 欄），搭配 Stage 05 原始 Y；各檔先驗 SHA、相同鍵值與欄位順序。Stage 03 Git SHA 維持釘選，不重算資料、不重新貼標。仍僅 retrospective research，禁止 publication 或更動 Risk ACTIVE。


In [ ]:
# 此設定每支 Notebook 可獨立執行；中途更換 Colab Runtime 也能接續同一研究 run。
import sys, os, json, hashlib, subprocess, traceback
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/給ai專用/janusChatGPT')
assert ROOT.is_dir(), '沒有找到給ai專用/janusChatGPT；不能改用個人用資料夾'
DATASET = ROOT/'資料源'/'JanusDataset_v3_202610'
MARKET = ROOT/'資料源'/'大盤產業_免費來源_2025H1_試跑'
RUN_HOME = ROOT/'模型研究'/'_colab_pipeline_runs'
ACTIVE = RUN_HOME/'ACTIVE_RUN.json'
REPO = Path('/content/janus-omniforge')
REPO_URL = 'https://github.com/tommylin15/janus-omniforge.git'
RUN_HOME.mkdir(parents=True, exist_ok=True)

def sh(*args, check=True, capture=False):
    return subprocess.run([str(x) for x in args],check=check,text=True,
                          capture_output=capture)

def sha_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for chunk in iter(lambda:f.read(2**20),b''):h.update(chunk)
    return h.hexdigest()

def write_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str,allow_nan=False)+'\n',encoding='utf-8')
    tmp.replace(path)

def read_json(path):return json.loads(Path(path).read_text(encoding='utf-8'))

def packages():
    # Colab 預裝 pandas/numpy/scikit-learn；只有缺少依賴才安裝。不引入付費服務。
    from importlib.util import find_spec
    missing=[(module,pip) for module,pip in [('pyarrow','pyarrow'),('sklearn','scikit-learn'),('xgboost','xgboost')]
             if find_spec(module) is None]
    if missing:sh(sys.executable,'-m','pip','install','-q',*[p for _,p in missing])
    import pandas, numpy, pyarrow, sklearn, xgboost
    return {k:__import__(k).__version__ for k in ['pandas','numpy','pyarrow','sklearn','xgboost']}

def repo_git(*args):
    # 優先使用公開存取；若 repo 是 private，使用 Colab Secrets 的 GITHUB_TOKEN。
    cmd=['git',*[str(x) for x in args]]
    env=dict(os.environ,GIT_TERMINAL_PROMPT='0')
    result=subprocess.run(cmd,env=env,text=True,capture_output=True)
    if result.returncode!=0:
        from google.colab import userdata
        token=userdata.get('GITHUB_TOKEN')
        if not token:raise RuntimeError('GIT_ACCESS_FAILED：repo 可能是 private；請設 Colab Secret GITHUB_TOKEN，或檢查存取權')
        import tempfile
        with tempfile.TemporaryDirectory() as folder:
            ask=Path(folder)/'askpass.sh'
            ask.write_text('#!/bin/sh\ncase "$1" in *Username*) printf x-access-token;; *) printf %s "$GITHUB_TOKEN";; esac\n')
            ask.chmod(0o700)
            env.update(GIT_ASKPASS=str(ask),GITHUB_TOKEN=token)
            result=subprocess.run(cmd,env=env,text=True,capture_output=True)
    if result.returncode:raise RuntimeError('GIT_COMMAND_FAILED：'+result.stderr[-500:])
    return result

def checkout(code_sha=None):
    if not (REPO/'.git').is_dir():repo_git('clone',REPO_URL,REPO)
    else:repo_git('-C',REPO,'fetch','origin','main')
    if code_sha:
        # 不能用最新 main 偷換固定研究程式版本。
        existing=sh('git','-C',REPO,'cat-file','-e',code_sha+'^{commit}',check=False)
        if existing.returncode:repo_git('-C',REPO,'fetch','origin',code_sha)
        repo_git('-C',REPO,'checkout','--detach','--force',code_sha)
    else:repo_git('-C',REPO,'checkout','--detach','--force','origin/main')
    actual=sh('git','-C',REPO,'rev-parse','HEAD',capture=True).stdout.strip()
    if code_sha and actual!=code_sha:raise RuntimeError('CODE_SHA_CHANGED')
    assert (REPO/'research/chatgpt/label_four_models.py').is_file()
    assert (REPO/'research/chatgpt/janus_train.py').is_file()
    code=str(REPO/'research/chatgpt')
    if code not in sys.path:sys.path.insert(0,code)
    return actual

def open_run():
    if not ACTIVE.is_file():raise RuntimeError('先執行 03_收檔驗收與同步.ipynb，建立研究 run')
    cfg=read_json(ACTIVE)
    out=RUN_HOME/cfg['run_id']
    if not out.is_dir():raise RuntimeError('RUN_FOLDER_MISSING')
    if not (DATASET/'canonical/CURRENT.json').is_file():raise RuntimeError('CURRENT_MISSING')
    if sha_file(DATASET/'canonical/CURRENT.json')!=cfg['current_sha256']:
        raise RuntimeError('CURRENT_CHANGED：來源版本已切換，請勿混用研究產物')
    manifest=DATASET/cfg['manifest_rel']
    if sha_file(manifest)!=cfg['manifest_sha256']:
        raise RuntimeError('MANIFEST_CHANGED：固定 release 有變動')
    checkout(cfg['git_sha'])
    return cfg,out

def save_stage(out,n,status,details):
    obj={'stage':n,'status':status,'updated_at_utc':datetime.now(timezone.utc).isoformat(),
         'publication_allowed':False,'details':details}
    write_json(Path(out)/f'{n}_status.json',obj)
    print(json.dumps(obj,ensure_ascii=False,indent=2,default=str)[:12000]);return obj

def stage(out,n):
    p=Path(out)/f'{n}_status.json'
    return read_json(p) if p.is_file() else {'stage':n,'status':'NOT_RUN'}

def ensure_stage(out,n,allow=('PASS',)):
    status=stage(out,n)['status']
    if status not in allow:raise RuntimeError(f'PREVIOUS_GATE_{n}={status}; allowed={allow}')

def keep_current(cfg):
    if sha_file(DATASET/'canonical/CURRENT.json')!=cfg['current_sha256']:
        raise RuntimeError('CURRENT_CHANGED_DURING_STAGE')

def fail_stage(out,n,exc):
    save_stage(out,n,'BLOCKED',{'error_type':type(exc).__name__, 'message':str(exc)[:1200]})
    raise exc

versions=packages()
print('Dependency versions:',versions)


In [ ]:
# 06｜以 Stage 05.1 擴充 X + Stage 05 原始 Y 研究訓練；保持 Stage 03 的 Git SHA 釘選
cfg,out=open_run();ensure_stage(out,'05',('PASS','PARTIAL'))
import json, hashlib, shutil, importlib
import pandas as pd
import numpy as np
import janus_train
import janus_precision as precision

role='quant'
expected_slots={'quant':15,'risk':12,'fundamental':13}
keys=['market','symbol','signal_group']
max_trees=400
train_contract='05.1.1-extended-x-original-y-v1'

try:
    status_path=out/'05.1_status.json'
    assert status_path.is_file(), 'STAGE05_1_NOT_RUN'
    candidate_status=read_json(status_path)
    assert candidate_status.get('status')=='PASS_RESEARCH_ONLY', 'STAGE05_1_NOT_PASSED'
    assert candidate_status.get('version')=='05.1.1', 'STAGE05_1_VERSION_MISMATCH_RERUN_05_1'
    assert candidate_status.get('publication_allowed') is False, 'UNSAFE_PUBLICATION_GATE'
    source=Path(candidate_status['outputs']).resolve()
    assert source.is_relative_to((out/'05_1_candidates').resolve()), 'UNSAFE_05_1_SOURCE'
    cm=read_json(source/'manifest.json')
    assert cm['fingerprint']==candidate_status['fingerprint'], 'STAGE05_1_FINGERPRINT_DRIFT'
    assert cm.get('training_contract')=='STAGE05_1_X_EXTENDED_WITH_IMMUTABLE_STAGE05_Y', 'TRAIN_CONTRACT_MISMATCH'
    assert cm.get('publication_allowed') is False, 'CANDIDATE_PUBLICATION_FORBIDDEN'
    assert cm['inputs']['active_run_git_sha']==cfg['git_sha'], 'PINNED_CODE_SHA_DRIFT'
    assert cm['inputs']['current_sha256']==cfg['current_sha256'], 'CURRENT_SOURCE_DRIFT'
    assert cm['inputs']['manifest_sha256']==cfg['manifest_sha256'], 'SOURCE_MANIFEST_DRIFT'
    for rel,checksum in cm['files'].items():
        file=(source/rel).resolve()
        assert file.is_relative_to(source) and file.is_file() and sha_file(file)==checksum, f'05_1_OUTPUT_DRIFT: {rel}'
    for rel,checksum in cm['inputs']['stage05'].items():
        file=(out/'xy'/rel if '/' in rel else out/rel).resolve()
        assert file.is_relative_to(out.resolve()) and file.is_file() and sha_file(file)==checksum, f'05_INPUT_DRIFT: {rel}'
    for entry in cm['inputs']['sources'].values():
        file=Path(entry['path']).resolve()
        assert file.is_relative_to((out/'labels').resolve()) and sha_file(file)==entry['sha256'], 'FACTOR_SOURCE_SHA_DRIFT'
    slots=read_json(out/'05_feature_slots.json')
    assert slots[role]['status']=='RESEARCH_DERIVED', f'ROLE_XY_NOT_READY: {role}'
    receipt=read_json(source/role/'receipt.json')
    assert receipt['role']==role and receipt['status']=='RESEARCH_CANDIDATE_DERIVED'
    assert receipt['extended_slots']==expected_slots[role], 'ROLE_SLOT_COUNT_MISMATCH'
    assert receipt.get('Y_reused_from_stage05') is True, 'Y_REUSE_CONTRACT_REQUIRED'
    features=list(receipt['feature_order'])
    assert len(features)==expected_slots[role]
    assert features==list(slots[role]['features'])+list(receipt['candidate_features']), 'FEATURE_ORDER_DRIFT'
    x_file=source/role/'X_extended.parquet'
    y_file=out/'xy'/role/'Y_NOT_X.parquet'
    assert receipt['X_extended_relpath']==f'{role}/X_extended.parquet'
    assert receipt['Y_stage05_relpath']==f'xy/{role}/Y_NOT_X.parquet'
    assert sha_file(x_file)==receipt['X_extended_sha256']==cm['files'][f'{role}/X_extended.parquet'], 'EXTENDED_X_SHA_DRIFT'
    assert sha_file(y_file)==receipt['original_y_sha256']==slots[role]['Y_sha256'], 'BASELINE_Y_SHA_DRIFT'
    x=pd.read_parquet(x_file); y=pd.read_parquet(y_file)
    assert list(x.columns)==keys+features, 'EXTENDED_X_SCHEMA_DRIFT'
    assert len(x)==len(y) and len(x)>0, 'X_Y_ROW_COUNT_MISMATCH'
    assert x[keys].equals(y[keys]), 'X_Y_KEY_ORDER_MISMATCH'
    assert not x.duplicated(keys).any() and not y.duplicated(keys).any(), 'X_Y_DUPLICATE_KEY'
    assert not any(c.startswith(('future_','forward_','target_','label_','y_')) for c in features), 'Y_LEAKAGE_IN_X'
    assert {'signal_at','mature_at','reason','y'}.issubset(y.columns), 'Y_TARGET_OR_TIME_MISSING'
    frame=pd.concat([x,y.drop(columns=keys)],axis=1)
    config={'role':role,'stage05_1_fingerprint':cm['fingerprint'],
            'x_sha256':receipt['X_extended_sha256'],'y_sha256':receipt['original_y_sha256'],
            'feature_order':features,'train_contract':train_contract,
            'train_engine_sha256':sha_file(janus_train.__file__),
            'precision_sha256':sha_file(precision.__file__),
            'git_sha':cfg['git_sha'],'max_trees':max_trees,'seed':janus_train.SEED,
            'dependencies':{k:importlib.import_module(k).__version__ for k in ('numpy','pandas','sklearn','xgboost','pyarrow')},
            'publication_allowed':False}
    fingerprint=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:16]
    model_dir=out/'training_05_1'/role/('candidate-'+fingerprint)
    marker=model_dir/'artifact_manifest.json'
    if marker.is_file():
        model_manifest=read_json(marker)
        assert model_manifest.get('fingerprint')==fingerprint, 'TRAIN_CACHE_FINGERPRINT_DRIFT'
        for rel,checksum in model_manifest['files'].items():
            file=(model_dir/rel).resolve()
            assert file.is_relative_to(model_dir.resolve()) and sha_file(file)==checksum, f'TRAIN_CACHE_HASH_DRIFT: {rel}'
        report=read_json(model_dir/'reports/result.json')
    else:
        assert not model_dir.exists(), 'INCOMPLETE_TRAIN_DIR_REQUIRES_INSPECTION'
        for folder in ['control','data','reports','models']:
            (model_dir/folder).mkdir(parents=True,exist_ok=True)
        write_json(model_dir/'control/config.json',config)
        write_json(model_dir/'control/input_manifest.json',cm['inputs'])
        write_json(model_dir/'control/evaluation_plan.json',{
            'kind':'retrospective_challenger','baseline':'stage05_original_features',
            'source_X':'stage05.1/X_extended','source_Y':'stage05/Y_NOT_X',
            'chronological_split_and_maturity_purge':'janus_train.fit_one',
            'publication_allowed':False})
        shutil.copyfile(x_file,model_dir/'data/X.parquet')
        shutil.copyfile(y_file,model_dir/'data/Y_NOT_X.parquet')
        assert sha_file(model_dir/'data/X.parquet')==config['x_sha256']
        assert sha_file(model_dir/'data/Y_NOT_X.parquet')==config['y_sha256']
        result=janus_train.fit_one(frame,features,role,model_dir/'models/main','y',max_trees)
        report={'role':role,'state':'RESEARCH_EVALUATED' if result['state']=='RESEARCH_EVALUATED' else 'BLOCKED_OR_INSUFFICIENT',
                'models':{'main':result},'mode':'retrospective','pit_status':'NOT_VERIFIED',
                'retrospective':True,'publication_allowed':False,
                'source_stage':'05.1','candidate_features':receipt['candidate_features'],
                'feature_order':features,'raw_rows':len(frame),
                'reason_counts':{str(k):int(v) for k,v in frame.reason.value_counts().items()},
                'missing_features':{f:int(frame[f].isna().sum()) for f in features},
                'X_sha256':config['x_sha256'],'Y_sha256':config['y_sha256'],
                'stage05_1_fingerprint':cm['fingerprint'],'git_sha':cfg['git_sha']}
        write_json(model_dir/'reports/result.json',report)
        files={str(p.relative_to(model_dir)):sha_file(p) for p in model_dir.rglob('*') if p.is_file() and p!=marker}
        write_json(marker,{'files':files,'fingerprint':fingerprint,'publication_allowed':False,
                           'source_stage':'05.1','Y_reused_from_stage05':True})
    keep_current(cfg)
    assert sha_file(x_file)==config['x_sha256'] and sha_file(y_file)==config['y_sha256'], 'SOURCE_CHANGED_DURING_TRAIN'
    write_json(out/'06_train.json',{'role':role,'model_dir':str(model_dir),
               'report_state':report['state'],'result_file':str(model_dir/'reports/result.json'),
               'git_sha':cfg['git_sha'],'X_sha256':config['x_sha256'],'Y_sha256':config['y_sha256'],
               'feature_slots':len(features),'source_stage':'05.1','publication_allowed':False})
    eligible=report['state']=='RESEARCH_EVALUATED'
    save_stage(out,'06','PASS' if eligible else 'PARTIAL',
               {'role':role,'state':report['state'],'model_dir':str(model_dir),
                'source_stage':'05.1','feature_slots':len(features),
                'X_sha256':config['x_sha256'],'Y_sha256':config['y_sha256'],
                'models':{k:v.get('state') for k,v in report.get('models',{}).items()},
                'retrospective':True,'historical_pit':'NOT_VERIFIED'})
except Exception as exc:
    fail_stage(out,'06',exc)
